In [ ]:
import matplotlib.pyplot as pyplot
import pandas as pd
import geopandas as gpd
import numpy as np
from google.colab import drive

In [ ]:
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
nino = pd.read_csv("/content/drive/MyDrive/Nino/raw_data/nina34.csv")
nino["nino_3.4"] = nino["   Nino Anom 3.4 Index  using NOAA ERSST v6 from NCEI  missing value -99.99 https://psl.noaa.gov/data/timeseries/month/"]
nino = nino.drop(columns="   Nino Anom 3.4 Index  using NOAA ERSST v6 from NCEI  missing value -99.99 https://psl.noaa.gov/data/timeseries/month/")

# remove -9999.0 values with na
(nino["nino_3.4"]== -9999.0).sum()
nino["nino_3.4"] = nino["nino_3.4"].replace(-9999.0, np.nan)

# make sure all na values are congregated at the beginning and end of data
print(nino[nino["nino_3.4"].isna()]["Date"])

0      1948-01-01
1      1948-02-01
2      1948-03-01
3      1948-04-01
4      1948-05-01
5      1948-06-01
6      1948-07-01
7      1948-08-01
8      1948-09-01
9      1948-10-01
10     1948-11-01
11     1948-12-01
12     1949-01-01
13     1949-02-01
14     1949-03-01
15     1949-04-01
16     1949-05-01
17     1949-06-01
18     1949-07-01
19     1949-08-01
20     1949-09-01
21     1949-10-01
22     1949-11-01
23     1949-12-01
942    2026-07-01
943    2026-08-01
944    2026-09-01
945    2026-10-01
946    2026-11-01
947    2026-12-01
Name: Date, dtype: object


In [ ]:
nino.describe()

,nino_3.4
count,918.000000
mean,-0.173878
std,0.871854
min,-2.450000
25%,-0.780000
50%,-0.250000
75%,0.357500
max,2.720000


In [ ]:
# need to fix the date column to remove the day at the end; allows me to merge with soil and precip data later
nino

,Date,nino_3.4
0,1948-01-01,NaN
1,1948-02-01,NaN
2,1948-03-01,NaN
3,1948-04-01,NaN
4,1948-05-01,NaN
...,...,...
943,2026-08-01,NaN
944,2026-09-01,NaN
945,2026-10-01,NaN
946,2026-11-01,NaN


In [ ]:
nino = nino.rename(columns={"Date": "date"})
nino["date"] = pd.to_datetime(nino["date"]).dt.strftime("%Y-%m")
nino

,date,nino_3.4
0,1948-01,NaN
1,1948-02,NaN
2,1948-03,NaN
3,1948-04,NaN
4,1948-05,NaN
...,...,...
943,2026-08,NaN
944,2026-09,NaN
945,2026-10,NaN
946,2026-11,NaN


In [ ]:
# load the soil and precip data and join with nino 3.4 on date
precip_soil = pd.read_parquet("/content/drive/MyDrive/Nino/cleaned_data/precip_soil_dynamic.parquet")
precip_soil

,GEOID,date,volumetric_soil_water_layer_3,precip_prior3,precip_prior6
56,01001,1950-02,0.442949,NaN,NaN
3164,01001,1950-03,0.459730,NaN,NaN
6272,01001,1950-04,0.442017,NaN,NaN
9380,01001,1950-05,0.413145,0.291365,NaN
12488,01001,1950-06,0.372161,0.304924,NaN
...,...,...,...,...,...
2837240,56045,2026-02,0.192497,0.044310,0.141030
2840348,56045,2026-03,0.192373,0.036165,0.123029
2843456,56045,2026-04,0.190505,0.031929,0.121160
2846564,56045,2026-05,0.187651,0.059701,0.104011


In [ ]:
# save nino3.4 dataset
nino.to_parquet("/content/drive/MyDrive/Nino/cleaned_data/nino34_dynamic.parquet")

In [ ]:
# merge precip_soil and nino3.4
precip_soil_nino = precip_soil.merge(nino, on="date", how="left")
precip_soil_nino

,GEOID,date,volumetric_soil_water_layer_3,precip_prior3,precip_prior6,nino_3.4
0,01001,1950-02,0.442949,NaN,NaN,-1.69
1,01001,1950-03,0.459730,NaN,NaN,-1.42
2,01001,1950-04,0.442017,NaN,NaN,-1.54
3,01001,1950-05,0.413145,0.291365,NaN,-1.75
4,01001,1950-06,0.372161,0.304924,NaN,-1.27
...,...,...,...,...,...,...
2850031,56045,2026-02,0.192497,0.044310,0.141030,-0.27
2850032,56045,2026-03,0.192373,0.036165,0.123029,-0.01
2850033,56045,2026-04,0.190505,0.031929,0.121160,0.29
2850034,56045,2026-05,0.187651,0.059701,0.104011,0.90


In [ ]:
precip_soil_nino["nino_3.4"].isna().sum() # no NA nino values
precip_soil_nino["volumetric_soil_water_layer_3"].isna().sum() # no NA soil values
precip_soil_nino["precip_prior3"].isna().sum() # 9324; 3 x the number of counties, which makes sense

np.int64(9324)

In [ ]:
import pandas as pd

colspecs = [(0,4),(4,6),(6,8),(8,11),(11,13),(13,14),(14,15),(15,16),(16,18),(18,26)]
names = ['year','state','stfips','cofips','registry','race','origin','sex','age','pop']
usecols = ['year','stfips','cofips','pop']          # only what you need

chunks = []
reader = pd.read_fwf(
    '/content/drive/MyDrive/Nino/raw_data/us.1969_2024.20ages.adjusted.txt',
    colspecs=colspecs, names=names,
    dtype={'stfips':str,'cofips':str,'pop':int},
    chunksize=1_000_000            # read 1M rows at a time
)

for chunk in reader:
    chunk['GEOID'] = chunk['stfips'] + chunk['cofips']
    # collapse THIS chunk to county-year subtotals, keep only that small result
    # the raw df has each county's populations for a given year splitten up by race, sex, age etc to make multiple rows out of the data
    # combine them to make it more usable
    sub = chunk.groupby(['GEOID','year'])['pop'].sum().reset_index()
    chunks.append(sub)

# combine the chunk-subtotals, then sum again (a county-year may span chunk boundaries)
pop_totals = (pd.concat(chunks)
                .groupby(['GEOID','year'])['pop'].sum()
                .reset_index()
                .rename(columns={'pop':'population'}))

In [ ]:
print(pop_totals[pop_totals["GEOID"].str[:2] > "56"]["GEOID"].unique()) # find all codes above valid states
pop_totals = pop_totals[pop_totals["GEOID"] != "99999"]
pop_totals.sort_values(by="year")

['99999']


,GEOID,year,population
108873,37141,1969,18084
108985,37145,1969,25570
108929,37143,1969,8233
24774,13139,1969,57843
98234,33011,1969,215811
...,...,...,...
118392,39173,2024,133077
55,01001,2024,61464
168572,54073,2024,7358
108928,37141,2024,70077


In [ ]:
from numpy._core.numeric import astype
from numpy._core.defchararray import zfill
# use TIGER county dataset to get the area of each county, ALAND
# then get population density (pop / areaa)

counties = gpd.read_file("/content/drive/MyDrive/Nino/raw_data/county_boundaries/tl_2018_us_county.shp")
land = counties[["GEOID", "ALAND"]].copy()
land["GEOID"] = land["GEOID"].astype(str).str.zfill(5)
land["ALAND"] = land["ALAND"].astype(float)
land["area_sqkm"] = land["ALAND"] / 1_000_000 # go from m^2 to km^2

In [ ]:
# fixes the dakota county issue
pop_totals["GEOID"] = pop_totals["GEOID"].replace("46113", "46102")

density = pop_totals.merge(land[["GEOID", "area_sqkm"]], on="GEOID", how="left")
density["pop_density"] = density["population"] / density["area_sqkm"]
density.sort_values(by="pop_density")

,GEOID,year,population,area_sqkm,pop_density
4702,02290,2024,5099,377034.650847,0.013524
4701,02290,2023,5136,377034.650847,0.013622
4700,02290,2022,5165,377034.650847,0.013699
4699,02290,2021,5272,377034.650847,0.013983
4698,02290,2020,5303,377034.650847,0.014065
...,...,...,...,...,...
164312,51918,1975,820045,NaN,NaN
164313,51918,1976,837547,NaN,NaN
164314,51918,1977,839748,NaN,NaN
164315,51918,1978,854651,NaN,NaN


In [ ]:
missing = density[density["area_sqkm"].isna()]
print(missing["GEOID"].value_counts())
# missing values are either:
  # out of scope (like alaska), so ignore for now and drop later (9xx or 02xxx)
  # VA county issues, mostly pre-1996 so will be dropped later (51xxx)
  # or issues with a dakota county being renamed (46113) (fixed above)


GEOID
51917    56
02270    35
08911    33
08913    33
08912    33
08914    33
15900    31
04910    25
02201    24
02280    24
02900    21
02232    20
51911    13
51910    13
35910    13
36910    11
51914    11
51915    11
51916    11
51913    11
51918    11
02910     4
02010     4
02231     4
Name: count, dtype: int64


In [ ]:
density

,GEOID,year,population,area_sqkm,pop_density
0,01001,1969,23819,1539.602123,15.470880
1,01001,1970,24661,1539.602123,16.017775
2,01001,1971,25503,1539.602123,16.564669
3,01001,1972,27156,1539.602123,17.638323
4,01001,1973,28453,1539.602123,18.480749
...,...,...,...,...,...
174896,56045,2020,6817,6210.804116,1.097603
174897,56045,2021,6747,6210.804116,1.086333
174898,56045,2022,6872,6210.804116,1.106459
174899,56045,2023,6828,6210.804116,1.099375


In [ ]:
precip_soil_nino["year"] = precip_soil_nino["date"].str.slice(0,4).astype(int)
precip_soil_nino_density = precip_soil_nino.merge(density, on=["GEOID", "year"], how="left")
precip_soil_nino_density[precip_soil_nino_density["year"] == 2024]

,GEOID,date,volumetric_soil_water_layer_3,precip_prior3,precip_prior6,nino_3.4,year,population,area_sqkm,pop_density
887,01001,2024-01,0.448493,0.217338,0.429919,1.81,2024,61464.0,1539.602123,39.922003
888,01001,2024-02,0.487520,0.353805,0.492330,1.52,2024,61464.0,1539.602123,39.922003
889,01001,2024-03,0.474603,0.379336,0.566970,1.13,2024,61464.0,1539.602123,39.922003
890,01001,2024-04,0.458295,0.420485,0.637823,0.77,2024,61464.0,1539.602123,39.922003
891,01001,2024-05,0.434250,0.377482,0.731287,0.23,2024,61464.0,1539.602123,39.922003
...,...,...,...,...,...,...,...,...,...,...
2850013,56045,2024-08,0.200866,0.144596,0.242912,-0.12,2024,6866.0,6210.804116,1.105493
2850014,56045,2024-09,0.196096,0.080136,0.252945,-0.26,2024,6866.0,6210.804116,1.105493
2850015,56045,2024-10,0.193047,0.049822,0.238152,-0.27,2024,6866.0,6210.804116,1.105493
2850016,56045,2024-11,0.191737,0.055402,0.199998,-0.25,2024,6866.0,6210.804116,1.105493


In [ ]:
# now add poverty rate data, updated for every county on the decade
import requests

KEY = "generate individual census key"
url = "https://api.census.gov/data/timeseries/poverty/saipe"

frames = []
for yr in range(1995, 2026):
  # SAEPOVRTALL_PT, poverty rate for all ages
  params = {
      "get": "NAME,SAEPOVRTALL_PT",
      "for": "county:*",
      "time": str(yr),
      "key":KEY
  }

  r = requests.get(url,params=params)
  if r.status_code != 200:
    print(yr, "failed:", r.text[:200]); continue
  rows = r.json() # first row is headers, rest is data
  df= pd.DataFrame(rows[1:], columns=rows[0])
  df["year"] = yr
  frames.append(df)

poverty_df = pd.concat(frames, ignore_index=True)
poverty_df

2025 failed: 


,NAME,SAEPOVRTALL_PT,time,state,county,year
0,Autauga County,13.4,1995,01,001,1995
1,Baldwin County,12.4,1995,01,003,1995
2,Barbour County,24.9,1995,01,005,1995
3,Bibb County,18.1,1995,01,007,1995
4,Blount County,13.7,1995,01,009,1995
...,...,...,...,...,...,...
94256,Sweetwater County,9.1,2024,56,037,2024
94257,Teton County,5.4,2024,56,039,2024
94258,Uinta County,10.7,2024,56,041,2024
94259,Washakie County,11.4,2024,56,043,2024


In [ ]:
poverty_df["GEOID"] = poverty_df["state"].astype(str).str.zfill(2) + poverty_df["county"].astype(str).str.zfill(3)
poverty_df["poverty_rate"] = pd.to_numeric(poverty_df["SAEPOVRTALL_PT"])
poverty_df["year"] = poverty_df["year"].astype(int)

poverty_df = poverty_df[["GEOID", "poverty_rate", "year"]]
poverty_df

,GEOID,poverty_rate,year
0,01001,13.4,1995
1,01003,12.4,1995
2,01005,24.9,1995
3,01007,18.1,1995
4,01009,13.7,1995
...,...,...,...
94256,56037,9.1,2024
94257,56039,5.4,2024
94258,56041,10.7,2024
94259,56043,11.4,2024


In [ ]:
print("total NaN poverty rows:", poverty_df["poverty_rate"].isna().sum())
# which YEARS are the NaNs in?
print(poverty_df[poverty_df["poverty_rate"].isna()]["year"].value_counts().sort_index())

total NaN poverty rows: 3162
year
1996    3141
2004       1
2005       1
2006       1
2007       1
2008       1
2009       1
2010       1
2011       1
2012       1
2013       1
2014       1
2015       1
2016       1
2017       1
2018       1
2019       1
2020       1
2021       1
2022       1
2023       1
2024       1
Name: count, dtype: int64


In [ ]:
# since almost all Nan's are in 1996, just replace all 1996 values with 1997
vals_1997 = poverty_df[poverty_df["year"] == 1997].set_index("GEOID")["poverty_rate"]

mask = poverty_df["year"] == 1996
poverty_df.loc[mask, "poverty_rate"] = poverty_df.loc[mask, "GEOID"].map(vals_1997)
print(poverty_df[poverty_df["poverty_rate"].isna()]["year"].value_counts().sort_index())

year
1996    2
2004    1
2005    1
2006    1
2007    1
2008    1
2009    1
2010    1
2011    1
2012    1
2013    1
2014    1
2015    1
2016    1
2017    1
2018    1
2019    1
2020    1
2021    1
2022    1
2023    1
2024    1
Name: count, dtype: int64


In [ ]:
# and finally merge them all together
precip_soil_nino_density_poverty = precip_soil_nino_density.merge(poverty_df, on=["GEOID", "year"], how="left")
precip_soil_nino_density_poverty[precip_soil_nino_density_poverty["year"] == 2024]

,GEOID,date,volumetric_soil_water_layer_3,precip_prior3,precip_prior6,nino_3.4,year,population,area_sqkm,pop_density,poverty_rate
887,01001,2024-01,0.448493,0.217338,0.429919,1.81,2024,61464.0,1539.602123,39.922003,12.9
888,01001,2024-02,0.487520,0.353805,0.492330,1.52,2024,61464.0,1539.602123,39.922003,12.9
889,01001,2024-03,0.474603,0.379336,0.566970,1.13,2024,61464.0,1539.602123,39.922003,12.9
890,01001,2024-04,0.458295,0.420485,0.637823,0.77,2024,61464.0,1539.602123,39.922003,12.9
891,01001,2024-05,0.434250,0.377482,0.731287,0.23,2024,61464.0,1539.602123,39.922003,12.9
...,...,...,...,...,...,...,...,...,...,...,...
2850013,56045,2024-08,0.200866,0.144596,0.242912,-0.12,2024,6866.0,6210.804116,1.105493,10.5
2850014,56045,2024-09,0.196096,0.080136,0.252945,-0.26,2024,6866.0,6210.804116,1.105493,10.5
2850015,56045,2024-10,0.193047,0.049822,0.238152,-0.27,2024,6866.0,6210.804116,1.105493,10.5
2850016,56045,2024-11,0.191737,0.055402,0.199998,-0.25,2024,6866.0,6210.804116,1.105493,10.5


In [ ]:
precip_soil_nino_density_poverty.to_parquet("/content/drive/MyDrive/Nino/intermediate_merged_data/precip_soil_nino_density_poverty.parquet")

In [ ]:
static = pd.read_parquet("/content/drive/MyDrive/Nino/intermediate_merged_data/updated_static_all.parquet")
df = precip_soil_nino_density_poverty.merge(static, on="GEOID", how="left")
df[df["year"] == 2024]

,GEOID,date,volumetric_soil_water_layer_3,precip_prior3,precip_prior6,nino_3.4,year,population,area_sqkm,pop_density,poverty_rate,hand_vulnerability,dist_coast_m,pct_no_vehicle,lat,lon
887,01001,2024-01,0.448493,0.217338,0.429919,1.81,2024,61464.0,1539.602123,39.922003,12.9,0.275304,2.154338e+05,0.056405,32.534912,-86.642761
888,01001,2024-02,0.487520,0.353805,0.492330,1.52,2024,61464.0,1539.602123,39.922003,12.9,0.275304,2.154338e+05,0.056405,32.534912,-86.642761
889,01001,2024-03,0.474603,0.379336,0.566970,1.13,2024,61464.0,1539.602123,39.922003,12.9,0.275304,2.154338e+05,0.056405,32.534912,-86.642761
890,01001,2024-04,0.458295,0.420485,0.637823,0.77,2024,61464.0,1539.602123,39.922003,12.9,0.275304,2.154338e+05,0.056405,32.534912,-86.642761
891,01001,2024-05,0.434250,0.377482,0.731287,0.23,2024,61464.0,1539.602123,39.922003,12.9,0.275304,2.154338e+05,0.056405,32.534912,-86.642761
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2866519,56045,2024-08,0.200866,0.144596,0.242912,-0.12,2024,6866.0,6210.804116,1.105493,10.5,0.258587,1.387298e+06,0.026453,43.840147,-104.567843
2866520,56045,2024-09,0.196096,0.080136,0.252945,-0.26,2024,6866.0,6210.804116,1.105493,10.5,0.258587,1.387298e+06,0.026453,43.840147,-104.567843
2866521,56045,2024-10,0.193047,0.049822,0.238152,-0.27,2024,6866.0,6210.804116,1.105493,10.5,0.258587,1.387298e+06,0.026453,43.840147,-104.567843
2866522,56045,2024-11,0.191737,0.055402,0.199998,-0.25,2024,6866.0,6210.804116,1.105493,10.5,0.258587,1.387298e+06,0.026453,43.840147,-104.567843
